In [3]:
## Script to prepare stimuli for Qualtrics survey for behavioural judgements of sentence similarity
# Run with python 3.10 fMRI env
# James Fodor 2023

import numpy as np
import json
import itertools
import sys
import os
import random

# load custom functions from library
sys.path.insert(0,'./..')
import voxels_processing as vp


In [ ]:
## Load sentence data and prepaire sentence pairs for survey

# Load full dataset
stimuli_path = "../Fodor2024-final108_sentences.txt"
all_sentences = np.loadtxt(stimuli_path, delimiter=',', dtype='str', encoding='utf-8')
all_sentences_indices = np.arange(len(all_sentences))
sent_ids_pairs_all = list(itertools.combinations(all_sentences_indices, 2))
num_stimuli = all_sentences_indices.shape[0]

# Get sentence pair IDs
sent_pair_ids_all = np.arange(len(sent_ids_pairs_all))
sent_pair_ids_dict = {x:idx for idx,x in enumerate(sent_ids_pairs_all)}
sent_pair_ids_diag = vp.get_pairs_in_diag(all_sentences_indices, num_sets=6)
sent_pair_ids_nondiag = [x for x in sent_pair_ids_all if x not in sent_pair_ids_diag]

# Print results
print('Total sent pairs:',len(sent_pair_ids_all))
print('Diag sent pairs:',len(sent_pair_ids_diag))
print('Non-diag sent pairs:',len(sent_pair_ids_nondiag))

Total sent pairs: 5778
Diag sent pairs: 918
Non-diag sent pairs: 4860


In [6]:
## Save sentence pairs with the formatting needed for qualtrics survey

# Select which subset of sentence pairs to work with
set_to_save = 'nondiags'
if set_to_save=='diags':
    sent_set_to_use = sent_pair_ids_diag # choose which subset of sentences to produce files for
    num_sets = 6 # how many files to divide this set of sentences into
elif set_to_save=='nondiags':
    sent_set_to_use = sent_pair_ids_nondiag
    num_sets = 10
sents_per_set = int(len(sent_set_to_use)/num_sets)

# Shuffle sentences so we don't show them in order in Qualtrics
sent_set_to_use_shuffled = sent_set_to_use.copy()
random.shuffle(sent_set_to_use_shuffled)

# Keep count to make sure we don't miss any
full_count = {}

# Loop over all sentence pairs and save to files
for set_num in np.arange(num_sets): # print 1/num_sets at a time as Qualtrics is dumb
    # Get subset of sentences
    set = sent_set_to_use_shuffled[sents_per_set*set_num:sents_per_set*(set_num+1)]
    
    # Prepare save file
    save_file_path = "../Fodor2024-final108_qualtrics-"+set_to_save+"-"+str(set_num+1)+".txt"
    save_file = open(save_file_path, "a", encoding='utf-8')

    # loop over sent pairs in subset and save to file
    for pair_of_sent_ids in np.array(sent_ids_pairs_all)[set]:
        sent_pair_id = sent_pair_ids_dict[tuple(pair_of_sent_ids)]
        sent_1 = all_sentences[pair_of_sent_ids[0]]
        sent_2 = all_sentences[pair_of_sent_ids[1]]
        rand_num = random.random()
        if rand_num>0.5: # randomise the order of sentence display
            save_file.writelines(str(sent_pair_id)+'. '+sent_1+'<br>'+sent_2+'\n\n')
        else:
            save_file.writelines(str(sent_pair_id)+'. '+sent_2+'<br>'+sent_1+'\n\n')
        save_file.writelines('1 - least similar meaning\n')
        save_file.writelines('2\n3\n4\n5\n6\n')
        save_file.writelines('7 - most similar meaning\n\n')
        full_count[sent_pair_id] = 1
    save_file.close()
    
# Check we got them all
print(np.sum(list(full_count.values())))

4860
